# 03 — Normalization + HVG selection

Log-normalize counts, select highly variable genes (HVGs), force-include ARC.

**Input**: `02_qc_filtered.h5ad`  
**Output**: `03_normalized_hvg.h5ad`

### In DEV_MODE

We subsample to `CELLS_PER_PATIENT_DEV` cells per patient (default: 1500) **after** QC but **before** normalization. This keeps the pipeline memory-light on an 8 GB Mac while preserving all 26 patients and all subtypes.

### Why seurat_v3 flavor for HVG?

It uses a variance-stabilizing transformation on raw counts and works well with sparse data. It correctly accounts for the mean-variance relationship in UMI counts.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import scanpy as sc

from src import config as cfg
from src import preprocessing as pp

sc.settings.verbosity = 2

adata = sc.read_h5ad(cfg.H5AD_QC)
print(f'Loaded {adata.n_obs} cells x {adata.n_vars} genes')
print(f'DEV_MODE = {cfg.DEV_MODE}')

In [ ]:
# Subsample in DEV_MODE
if cfg.DEV_MODE:
    adata = pp.stratified_subsample(
        adata,
        group_key=cfg.BATCH_KEY,
        n_per_group=cfg.CELLS_PER_PATIENT_DEV,
        random_state=cfg.RANDOM_SEED,
    )
print(f'After subsampling: {adata.n_obs} cells')

In [ ]:
# Normalize + log1p. Raw counts stored in layers['counts'].
adata = pp.normalize_log(
    adata,
    target_sum=cfg.NORMALIZE_TOTAL,
    store_raw_counts_layer=True,
)
print('Layers:', list(adata.layers.keys()))

In [ ]:
# HVG selection. We use seurat_v3 which requires raw counts.
# ARC is forced into the HVG set via protected_genes.
adata = pp.select_hvgs(
    adata,
    n_top_genes=cfg.N_HVGS,
    flavor='seurat_v3',
    batch_key=None,
    protected_genes=cfg.PROTECTED_GENES,
)

# Sanity check: ARC flagged as HVG?
if 'ARC' in adata.var_names:
    print(f"ARC in HVGs: {bool(adata.var.loc['ARC', 'highly_variable'])}")

In [ ]:
# Save
adata.write_h5ad(cfg.H5AD_NORM, compression='gzip')
print(f'Saved: {cfg.H5AD_NORM} ({cfg.H5AD_NORM.stat().st_size / 1e6:.1f} MB)')

---

**Next**: `04_integration_clustering.ipynb`